# 14 — Assignment toolkit: files, range image, normals, components

**Needed for the assignment:** $\bigstar$ YES. **Study:** 2 h, **Practice:** 1 h.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Idea

Topics 01–08 give you the fitting and RANSAC. This topic covers the extra pieces that made the solution work on *real* depth-camera data:

1.  read PCD and write PLY yourself;

2.  put the points back on an image grid (the **range image**), so every point has neighbours;

3.  from the neighbours, get a **surface normal** per point;

4.  measure the **noise** robustly;

5.  split a plane into **connected components**.

## Key formulas

**PCD header** (text): `FIELDS x y z rgb intensity`, `SIZE`, `TYPE`, `COUNT`, `POINTS N`, then `DATA ascii` or `binary`. Read only $x,y,z$, and drop NaN and $(0,0,0)$ points.\
**PLY** (ASCII): `ply / format ascii 1.0 / element vertex N / property float x,y,z / property uchar red,green,blue / end_header`, then one line "`x y z r g b`" per point.\
**Range image:** $u=x/z,\ v=y/z$ (topic 10 with $K=I$), scaled onto a grid of about $0.6N$ cells.\
**Local normal** over a $k\times k$ window, from sums (computed with `cv2.boxFilter`): $$\boldsymbol\mu=\tfrac1n\sum\mathbf{p},\qquad C=\tfrac1n\sum\mathbf{p}\mathbf{p}^T-\boldsymbol\mu\boldsymbol\mu^T,\qquad \mathbf{n}=\text{smallest eigenvector of }C$$ **Robust noise:** $\hat\sigma=1.4826\cdot\text{median}|r_i|$ (equals the std for Gaussian noise, and ignores outliers).\
**Inlier tests:**

-   plane: $|\mathbf{n}^T\mathbf{p}+d|<\tau$ *and* $|\mathbf{n}_i^T\mathbf{n}|>\cos25^\circ$;

-   sphere: $|\|\mathbf{p}-\mathbf{c}\|-r|<\tau_s$ *and* $\mathbf{n}_i\parallel(\mathbf{p}-\mathbf{c})$.

**Components:** `cv2.connectedComponents(mask, connectivity=8)` on the grid cells that hold inliers.

# Worked example 1 (on paper): normal from sums

Window points $(0,0,0),(2,0,0),(0,2,0),(2,2,0)$, $n=4$.\
$\boldsymbol\mu=(1,1,0)$.  $\tfrac1n\sum x^2=\tfrac{0+4+0+4}4=2\Rightarrow C_{xx}=2-1=1$.  $\tfrac1n\sum xy=\tfrac{0+0+0+4}4=1\Rightarrow C_{xy}=1-1=0$.  $C_{yy}=1$. All $z$ terms are $0$. $$C=\begin{bmatrix}1&0&0\\0&1&0\\0&0&0\end{bmatrix}\ \Rightarrow\ \lambda_{\min}=0,\quad\mathbf{n}=(0,0,1)\ ✓\ (\text{the points lie in the plane }z=0)$$

# Worked example 2 (on paper): robust noise

Residuals $0.9,-1.1,1.0,-0.8,1.2,-1.0,40$. Sorted $|r|$: $0.8,0.9,1.0,1.0,1.1,1.2,40$, so the median is $1.0$ and $\hat\sigma=1.483$. The ordinary std is $14.0$: a single outlier ruins it.

# The same in Python / NumPy + OpenCV (a tiny synthetic depth camera: wall and ball)

In [2]:
import numpy as np
import cv2

# ---------- 1) a tiny synthetic depth camera: a wall at z = 5 and a ball in front ----------
H, W = 60, 80
v, u = np.mgrid[0:H, 0:W]
x_n, y_n = (u - W / 2) / 60.0, (v - H / 2) / 60.0          # normalized image coordinates
ray = np.dstack([x_n, y_n, np.ones_like(x_n)])               # ray direction per pixel

centre, r = np.array([0.0, 0.0, 4.7]), 0.8      # back of the ball reaches the wall band
# ray-sphere intersection: |s*ray - c|^2 = r^2  ->  a s^2 - 2 b s + (|c|^2 - r^2) = 0
a = np.sum(ray**2, axis=2); bq = ray @ centre; cq = centre @ centre - r**2
disc = bq**2 - a * cq
s_ball = (bq - np.sqrt(np.clip(disc, 0, None))) / a
s = np.where(disc > 0, s_ball, 5.0)                          # wall: z = 5  ->  s = 5
P = ray * s[..., None] + np.random.default_rng(0).normal(0, 0.005, (H, W, 3))
is_ball = disc > 0
print("pixels on ball:", is_ball.sum(), " on wall:", (~is_ball).sum())

# ---------- 2) normals: window sums with cv2.boxFilter -> covariance -> smallest eigvec ----------
k = 5
box = lambda img: cv2.boxFilter(img, -1, (k, k), normalize=True, borderType=cv2.BORDER_REPLICATE)
mu = np.dstack([box(P[..., i]) for i in range(3)])
cov = np.empty((H, W, 3, 3))
for i in range(3):
    for j in range(3):
        cov[..., i, j] = box(P[..., i] * P[..., j]) - mu[..., i] * mu[..., j]  # E[xy] - E[x]E[y]
val, vec = np.linalg.eigh(cov)                               # sorted ascending
normal = vec[..., :, 0]

# ---------- 3) "points that look like the wall plane": close AND parallel normal ----------
dist_ok = np.abs(P[..., 2] - 5.0) < 0.7                      # loose threshold on purpose
angle_ok = np.abs(normal @ np.array([0, 0, 1.0])) > np.cos(np.radians(25))
print("ball pixels passing distance only      :", (is_ball & dist_ok).sum())
print("ball pixels passing distance AND normal:", (is_ball & dist_ok & angle_ok).sum())

# ---------- 4) connected components of the 'wall' mask ----------
mask = (dist_ok & angle_ok).astype(np.uint8) * 255
n, cc = cv2.connectedComponents(mask, connectivity=8)
sizes = np.bincount(cc.ravel())[1:]
print("components:", n - 1, " sizes:", sizes)

# ---------- 5) robust noise estimate: 1.4826 * median |r| ----------
res = np.array([0.9, -1.1, 1.0, -0.8, 1.2, -1.0, 40.0])     # one huge outlier
print("std =", round(res.std(), 2), "  robust sigma =", round(1.4826 * np.median(np.abs(res)), 3))

pixels on ball: 341  on wall: 4459
ball pixels passing distance only      : 48
ball pixels passing distance AND normal: 0
components: 1  sizes: [4275]
std = 14.02   robust sigma = 1.483


With only a distance test, 48 ball pixels would be taken by the wall plane. The normal test removes all of them. This is the same "ring cut out of the ball" problem seen on the real frame.

# The pipeline, and why each step exists (what failed without it)

1.  **Plain sequential plane RANSAC** cut rings and flat caps out of the ball. **Fix:** normal test, and find the sphere *first*.

2.  **Threshold from local noise** ($\approx1$ mm) was too tight, because big far walls bend by up to $10$ mm (sensor bias), so the walls broke into fragments. **Fix:** plane threshold $=3\hat\sigma$ of the dominant plane's residuals ($\approx13$ mm).

3.  **Sphere RANSAC with 4 random points** needs about $737\,000$ iterations when $w\approx0.05$ (topic 08). **Fix:** *local sampling*, i.e. a seed point plus 3 grid neighbours.

4.  **Large tangent spheres** touching a wall competed with the ball. **Fix:** a tight sphere threshold ($3\times$ local noise) for the search, then the looser threshold to collect the noisy rim.

5.  **One RANSAC plane** also took coplanar points elsewhere in the room. **Fix:** connected components; drop tiny pieces, and merge a plane that is found twice.

## Practice

Write your solution in the empty cell under each exercise. The **expected result** tells you what you should get, not how to get it. Grey *provided data* cells are already filled in; run them first.

### Exercise 1

Write `read_pcd` for ASCII files: parse the header until `DATA`, find the column indices of `x y z` from `FIELDS`, drop NaN and $(0,0,0)$ points. Test it on Frame4994.

> **Expected result:** Shape `(215659, 3)` (217088 − 1429 invalid points), $z$ between $\approx259.4$ and $\approx1779.5$ (mm).

In [3]:
# --- provided data (run me first) ---
# PCD_PATH : path to the frame (put the .pcd file next to this notebook)
# Header of Frame4994: FIELDS x y z rgb intensity, POINTS 217088, DATA ascii
# Edge cases: invalid points are stored as (0, 0, 0) and must be removed.
PCD_PATH = 'Frame4994.pcd'

In [4]:
# Exercise 1: your solution


### Exercise 2

Write `write_ply` (ASCII, x y z red green blue). Colour all points with $z<1000$ red and the rest grey, and open the result in MeshLab or CloudCompare.

> **Expected result:** 59041 red points; the near part of the scene (the ball and the object on the left) appears red.

In [5]:
# Exercise 2: your solution


### Exercise 3

Build the range image of Frame4994 ($u=x/z$, $v=y/z$, grid with about $0.6N$ cells) and save the depth as a PNG. Can you see the ball?

> **Expected result:** Grid $395\times328$ (W × H). The ball is a round blob a bit below the image centre, nearer than the walls behind it.

In [6]:
# Exercise 3: your solution


### Exercise 4

**The assignment:** write it yourself, following the steps in topic 14. Test it on Frame5922 and Frame7183, and compare with the given solution only at the end.

> **Expected result:** On Frame4994: exactly one sphere, $r\approx216$ mm, centre $\approx(-81,-20,1270)$ mm, about 12–13k points, plus around 10–15 planar segments.

In [7]:
# Exercise 4: your solution
